### IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore") # Suppress warnings for cleaner output

### LOAD DATA

In [ ]:
df = pd.read_csv("../data/household_power_consumption.txt", sep=";", na_values=["?"], low_memory=False)
df["datetime"] = pd.to_datetime(df["Date"] + " " + df["Time"], dayfirst=True)
df = df.drop(columns=["Date", "Time"])
df = df.set_index("datetime").sort_index()
df.head()

hourly_power = df["Global_active_power"].resample("h").mean()
print("hourly_power_shape", hourly_power.shape)

### ANOMALY DETECTION FUNCTION

In [ ]:
def detect_anomalies(series, rolling_window=24, spike_std=3, flat_var=0.001, flat_window=6):
    """
    Detects and labels anomalies in a time series.
    Labels: 0=normal, 1=dropout, 2=spike, 3=flatline
    """

    result = pd.DataFrame({'value': series})
    result['label'] = 0
    result['anomaly_type'] = 'normal'

    # 1. Detect missing values (dropouts)
    mask_missing = result['value'].isna()
    result.loc[mask_missing, 'label'] = 1
    result.loc[mask_missing, 'anomaly_type'] = 'dropout'

    # 2. Detect spikes using median + MAD (robust to outliers in the window)
    rolling_median = result['value'].rolling(window=rolling_window, center=False, min_periods=1).median()
    rolling_mad = result['value'].rolling(window=rolling_window, center=False, min_periods=1).apply(
        lambda x: np.median(np.abs(x - np.median(x)))
    )
    # 0.6745 scales MAD to be comparable to std under a normal distribution
    robust_std = rolling_mad / 0.6745
    mask_spike = (abs(result['value'] - rolling_median) > (spike_std * robust_std)) & ~mask_missing
    result.loc[mask_spike, 'label'] = 2
    result.loc[mask_spike, 'anomaly_type'] = 'spike'

    # 3. Detect flatlines
    rolling_var = result['value'].rolling(window=flat_window, min_periods=1).var()
    mask_flat = (rolling_var < flat_var) & ~mask_missing & (result['value'] > 0)
    result.loc[mask_flat, 'label'] = 3
    result.loc[mask_flat, 'anomaly_type'] = 'flatline'

    return result

anomalies = detect_anomalies(hourly_power)
print(anomalies['label'].value_counts())


### VISUALIZE ANOMALIES ON A SAMPLE MONTH

In [ ]:
import matplotlib.patches as mpatches

sample_month = anomalies.loc['2008-01']

# color map
color_map = {
    'normal': 'steelblue',
    'dropout': 'lightgray',
    'spike': 'red',
    'flatline': 'orange'
}

# plot the signal first
fig, ax = plt.subplots(figsize=(16, 4))
ax.plot(sample_month.index, sample_month['value'], color='steelblue', linewidth=0.8, zorder=1)

for anomaly_type, color in color_map.items():
    mask = sample_month['anomaly_type'] == anomaly_type
    if anomaly_type == 'normal':
        continue
    ax.scatter(sample_month.index[mask], sample_month['value'][mask], color=color, s=40, zorder=2, label=anomaly_type)

patches = [mpatches.Patch(color=color, label=anomaly_type) for anomaly_type, color in color_map.items() if anomaly_type != 'normal']
ax.legend(handles=patches)

ax.set_title("Hourly Power Consumption with Anomalies (January 2008)")
ax.set_xlabel("Date")
ax.set_ylabel("Global Active Power (kW)")
plt.tight_layout()
plt.show()


### Synthetic fault injection

In [ ]:
def inject_fault(series, fault_type='spike', start='2008-06-01', duration_hours=5):
    """
    Injects a synthetic fault into the series for demo purposes.
    fault_type: 'spike', 'flatline', or 'dropout'
    """
    injected = series.copy()
    idx = pd.date_range(start=start, periods=duration_hours, freq='h')
    idx = idx[idx.isin(series.index)]  # safety check

    if fault_type == 'spike':
        injected[idx] = series.mean() + 6 * series.std()
    elif fault_type == 'flatline':
        injected[idx] = series.median() * 0.01
    elif fault_type == 'dropout':
        injected[idx] = np.nan

    return injected

# Inject and re-run detection
injected_series = inject_fault(hourly_power, fault_type='spike', start='2008-06-01', duration_hours=5)
tagged_injected = detect_anomalies(injected_series)

# Check it was caught
window = tagged_injected['2008-06-01':'2008-06-02']
print(window[window['anomaly_type'] != 'normal'][['value', 'anomaly_type']])